# 09 - Desigualdad territorial ENIGH 2018-2024

Este notebook resume una etapa descriptiva de desigualdad territorial para la tesina. El objetivo es calcular Gini ponderado, compararlo con el benchmark de Banco de México cuando la definición sea compatible, y separar desigualdad dentro de territorios de brechas entre territorios.

No se crean modelos, no se deflacta, no se incorporan fuentes nuevas y no se aproximan zonas metropolitanas con municipios sueltos.

## Definición metodológica

Benchmark revisado: Banco de México, Recuadro 2 del Reporte sobre las Economías Regionales enero-marzo 2024, "Disminución de la desigualdad de ingresos regional en un contexto de crecimiento propobre: 2018-2022".

Punto clave: el recuadro indica que el Gini usa **ingreso corriente total promedio por hogar**. Por eso la comparación principal de este notebook usa:

- ingreso: `ing_cor_hogar_oficial_tri`;
- ponderador: `factor`;
- universo: todos los hogares con ingreso no faltante, no negativo y factor positivo;
- ceros: se conservan como valores válidos;
- escala: se reporta Gini en 0-1 y 0-100;
- años comparables con Banxico: 2018, 2020 y 2022.

La comparación no busca calzar perfectamente. Banco de México usa bases generadas por CONEVAL a partir de ENIGH; aquí se usa el mart propio con la variable oficial de `concentradohogar`. Diferencias pequeñas o moderadas pueden venir de definición CONEVAL, procesamiento, universo, ponderación, escala temporal del ingreso o redondeo.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (10, 5), "axes.titlesize": 13, "axes.labelsize": 11})

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "interim" / "revision_4").exists():
            return candidate
    raise FileNotFoundError("No se encontró data/interim/revision_4 desde el directorio actual.")

ROOT = find_project_root()
REV4 = ROOT / "data" / "interim" / "revision_4"
HOGAR_PATH = REV4 / "mart_hogar_2018_2024.csv.gz"
PERSONA_PATH = REV4 / "mart_persona_2018_2024.csv.gz"

WEIGHT = "factor"
INC_HH = "ing_cor_hogar_oficial_tri"
INC_PC = "ing_cor_pc_oficial_tri"
INC_LAB = "ingreso_persona_laboral_negocio_tri"
YEARS = [2018, 2020, 2022, 2024]

print(f"Proyecto: {ROOT}")
print(f"Base hogares: {HOGAR_PATH.exists()} | {HOGAR_PATH}")
print(f"Base personas: {PERSONA_PATH.exists()} | {PERSONA_PATH}")

## Carga de bases analíticas

Se cargan únicamente las columnas necesarias para esta revisión. La base de hogares se usa para Gini comparable con Banxico; la base de personas se usa solo para ingreso laboral individual en la sección CDMX.

In [ ]:
hogar_cols = [
    "anio", "region_banxico", "entidad", "cve_ent", "tam_loc_desc", "est_socio_desc",
    WEIGHT, "factor_hogar", "est_dis", "upm", INC_HH, INC_PC, "ingtrab_hogar_oficial_tri", "tot_integ",
]
persona_cols = [
    "anio", "region_banxico", "entidad", "cve_ent", "tam_loc_desc", "est_socio_desc",
    "sexo_desc", WEIGHT, INC_LAB, "ing_cor_hogar_pc_oficial_tri", "edad",
]
hogar = pd.read_csv(HOGAR_PATH, usecols=hogar_cols, low_memory=False)
persona = pd.read_csv(PERSONA_PATH, usecols=persona_cols, low_memory=False)

for col in ["anio", WEIGHT, "factor_hogar", INC_HH, INC_PC, "ingtrab_hogar_oficial_tri", "tot_integ"]:
    hogar[col] = pd.to_numeric(hogar[col], errors="coerce")
for col in ["anio", WEIGHT, INC_LAB, "ing_cor_hogar_pc_oficial_tri", "edad"]:
    persona[col] = pd.to_numeric(persona[col], errors="coerce")

validacion_base = pd.DataFrame([
    {"base": "hogar", "filas": len(hogar), "columnas": hogar.shape[1], "factor_missing": int(hogar[WEIGHT].isna().sum()), "ingreso_missing": int(hogar[INC_HH].isna().sum()), "ingreso_negativo": int(hogar[INC_HH].lt(0).sum())},
    {"base": "persona", "filas": len(persona), "columnas": persona.shape[1], "factor_missing": int(persona[WEIGHT].isna().sum()), "ingreso_laboral_missing": int(persona[INC_LAB].isna().sum()), "ingreso_laboral_negativo": int(persona[INC_LAB].lt(0).sum())},
])
display(validacion_base)

## Funciones ponderadas

La fórmula operativa del Gini usa la curva de Lorenz ponderada. Se ordena el ingreso de menor a mayor, se acumula población ponderada e ingreso ponderado, y se calcula:

```text
Gini = 1 - 2 * área bajo la curva de Lorenz
```

El cálculo conserva ceros y excluye únicamente ingresos faltantes, negativos o pesos no positivos.

In [ ]:
def weighted_quantile(values, weights, qs):
    x = pd.to_numeric(values, errors="coerce").to_numpy(dtype="float64")
    w = pd.to_numeric(weights, errors="coerce").to_numpy(dtype="float64")
    q = np.atleast_1d(qs).astype(float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[mask], w[mask]
    if len(x) == 0:
        return np.full(len(q), np.nan)
    order = np.argsort(x, kind="mergesort")
    x, w = x[order], w[order]
    cum_w = np.cumsum(w) / np.sum(w)
    return np.interp(q, cum_w, x, left=x[0], right=x[-1])

def weighted_gini(values, weights):
    x = pd.to_numeric(values, errors="coerce").to_numpy(dtype="float64")
    w = pd.to_numeric(weights, errors="coerce").to_numpy(dtype="float64")
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0) & (x >= 0)
    x, w = x[mask], w[mask]
    if len(x) == 0 or np.sum(x * w) <= 0:
        return np.nan
    order = np.argsort(x, kind="mergesort")
    x, w = x[order], w[order]
    cum_w = np.cumsum(w)
    cum_xw = np.cumsum(x * w)
    pop_share = np.insert(cum_w / cum_w[-1], 0, 0)
    income_share = np.insert(cum_xw / cum_xw[-1], 0, 0)
    return float(1 - 2 * np.trapezoid(income_share, pop_share))

def weighted_summary(df, group_cols, income_col, positive_only=False):
    work = df[df[income_col].gt(0)].copy() if positive_only else df.copy()
    rows = []
    for key, g in work.groupby(group_cols, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        row = dict(zip(group_cols, key))
        x = pd.to_numeric(g[income_col], errors="coerce")
        w = pd.to_numeric(g[WEIGHT], errors="coerce")
        mask = x.notna() & w.notna() & w.gt(0) & x.ge(0)
        x, w = x[mask], w[mask]
        qs = weighted_quantile(x, w, [0.25, 0.50, 0.75, 0.90, 0.95])
        row.update({
            "n": int(mask.sum()),
            "n_ponderado": float(w.sum()),
            "media": float(np.average(x, weights=w)) if len(x) else np.nan,
            "p25": qs[0],
            "mediana": qs[1],
            "p75": qs[2],
            "p90": qs[3],
            "p95": qs[4],
            "gini": weighted_gini(x, w),
        })
        rows.append(row)
    out = pd.DataFrame(rows)
    out["gini_0_100"] = out["gini"] * 100
    return out

def money_cols(df):
    cols = ["media", "p25", "mediana", "p75", "p90", "p95", "gini", "gini_0_100"]
    return df.round({c: 2 for c in cols if c in df.columns})

## Benchmark Banxico 2018-2022

Los valores de Banxico se capturan tal como aparecen en la tabla oficial, en escala 0-100.

In [ ]:
banxico_gini = pd.DataFrame([
    ("Norte", 2018, 43.1), ("Norte", 2020, 43.7), ("Norte", 2022, 40.3),
    ("Centro Norte", 2018, 43.2), ("Centro Norte", 2020, 41.3), ("Centro Norte", 2022, 40.4),
    ("Centro", 2018, 45.1), ("Centro", 2020, 44.5), ("Centro", 2022, 42.1),
    ("Sur", 2018, 47.5), ("Sur", 2020, 45.8), ("Sur", 2022, 44.9),
    ("Nacional", 2018, 45.7), ("Nacional", 2020, 45.0), ("Nacional", 2022, 43.1),
], columns=["region_banxico", "anio", "gini_banxico_0_100"])
display(banxico_gini)

## Gini nacional

Este es el cálculo principal con `ing_cor_hogar_oficial_tri`. La lectura temporal debe hacerse con cautela porque los ingresos están en pesos nominales; el Gini dentro de cada año no depende de multiplicar todos los ingresos por una constante, pero la comparación temporal sustantiva sí requiere más contexto.

In [ ]:
gini_nacional = weighted_summary(hogar, ["anio"], INC_HH)
display(money_cols(gini_nacional[["anio", "n", "n_ponderado", "media", "mediana", "gini", "gini_0_100"]]))

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.lineplot(data=gini_nacional, x="anio", y="gini_0_100", marker="o", ax=ax)
ax.set_title("Gini nacional ponderado: ingreso corriente total del hogar")
ax.set_xlabel("Año")
ax.set_ylabel("Gini (0-100)")
ax.set_xticks(YEARS)
display(fig)
plt.close(fig)

## Gini por región Banxico y validación

La validación compara dirección, orden relativo y magnitud aproximada. No se fuerza el resultado a coincidir con Banxico.

In [ ]:
gini_region = weighted_summary(hogar, ["anio", "region_banxico"], INC_HH)
gini_region_display = money_cols(gini_region[["anio", "region_banxico", "n", "media", "mediana", "gini_0_100"]].sort_values(["anio", "region_banxico"]))
display(gini_region_display)

propio_para_banxico = pd.concat([
    gini_region[["anio", "region_banxico", "gini_0_100"]],
    gini_nacional.assign(region_banxico="Nacional")[["anio", "region_banxico", "gini_0_100"]],
], ignore_index=True)
comparacion_banxico = propio_para_banxico.merge(banxico_gini, on=["anio", "region_banxico"], how="inner")
comparacion_banxico["dif_puntos"] = comparacion_banxico["gini_0_100"] - comparacion_banxico["gini_banxico_0_100"]
comparacion_banxico["abs_dif_puntos"] = comparacion_banxico["dif_puntos"].abs()
display(money_cols(comparacion_banxico.sort_values(["anio", "region_banxico"])))

discrepancia_max = comparacion_banxico.loc[comparacion_banxico["abs_dif_puntos"].idxmax()].copy()
display(Markdown(
    f"**Mayor discrepancia:** {discrepancia_max['region_banxico']} {int(discrepancia_max['anio'])}, "
    f"{discrepancia_max['dif_puntos']:.2f} puntos frente a Banxico."
))

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=gini_region, x="anio", y="gini_0_100", hue="region_banxico", marker="o", ax=ax)
ax.set_title("Gini por región Banxico")
ax.set_xlabel("Año")
ax.set_ylabel("Gini (0-100)")
ax.set_xticks(YEARS)
ax.legend(title="Región")
display(fig)
plt.close(fig)

## Gini por entidad, tamaño de localidad y estrato socioeconómico

Estas tablas miden desigualdad dentro de cada territorio o estrato. No son brechas entre territorios; por ejemplo, un Gini alto en una entidad indica mayor dispersión interna de ingresos de hogares dentro de esa entidad.

In [ ]:
gini_entidad = weighted_summary(hogar, ["anio", "entidad"], INC_HH)
gini_tam_loc = weighted_summary(hogar, ["anio", "tam_loc_desc"], INC_HH)
gini_est_socio = weighted_summary(hogar, ["anio", "est_socio_desc"], INC_HH)

entidad_2024 = gini_entidad[gini_entidad["anio"].eq(2024)].sort_values("gini_0_100")
display(Markdown("### Entidades con menor Gini interno en 2024"))
display(money_cols(entidad_2024.head(8)[["entidad", "n", "mediana", "gini_0_100"]]))
display(Markdown("### Entidades con mayor Gini interno en 2024"))
display(money_cols(entidad_2024.tail(8)[["entidad", "n", "mediana", "gini_0_100"]]))

display(Markdown("### Tamaño de localidad, 2024"))
display(money_cols(gini_tam_loc[gini_tam_loc["anio"].eq(2024)][["tam_loc_desc", "n", "media", "mediana", "gini_0_100"]].sort_values("mediana", ascending=False)))

display(Markdown("### Estrato socioeconómico, 2024"))
display(money_cols(gini_est_socio[gini_est_socio["anio"].eq(2024)][["est_socio_desc", "n", "media", "mediana", "gini_0_100"]].sort_values("mediana", ascending=False)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=gini_tam_loc, x="anio", y="mediana", hue="tam_loc_desc", marker="o", ax=axes[0])
axes[0].set_title("Mediana por tamaño de localidad")
axes[0].set_xlabel("Año")
axes[0].set_ylabel("Ingreso corriente hogar")
axes[0].legend(title="Tamaño", fontsize=8)
sns.lineplot(data=gini_est_socio, x="anio", y="mediana", hue="est_socio_desc", marker="o", ax=axes[1])
axes[1].set_title("Mediana por estrato socioeconómico")
axes[1].set_xlabel("Año")
axes[1].set_ylabel("Ingreso corriente hogar")
axes[1].legend(title="Estrato", fontsize=8)
fig.tight_layout()
display(fig)
plt.close(fig)

## CDMX

Para CDMX se calculan métricas descriptivas ponderadas por año. Los ingresos están en pesos nominales trimestrales. Para ingreso laboral individual se usa el universo con ingreso laboral positivo, porque la pregunta aquí es el monto entre quienes reportan ingreso laboral; la probabilidad de tener ingreso laboral positivo debe analizarse aparte.

In [ ]:
def cve_ent_2(series):
    return series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(2)

cdmx_hogar = hogar[cve_ent_2(hogar["cve_ent"]).eq("09")].copy()
cdmx_persona = persona[cve_ent_2(persona["cve_ent"]).eq("09")].copy()

cdmx_ingreso_hogar = weighted_summary(cdmx_hogar, ["anio"], INC_HH).assign(metrica="Ingreso corriente hogar")
cdmx_ingreso_pc = weighted_summary(cdmx_hogar, ["anio"], INC_PC).assign(metrica="Ingreso corriente per capita hogar")
cdmx_laboral = weighted_summary(cdmx_persona, ["anio"], INC_LAB, positive_only=True).assign(metrica="Ingreso laboral individual positivo")
cdmx_metricas = pd.concat([cdmx_ingreso_hogar, cdmx_ingreso_pc, cdmx_laboral], ignore_index=True)

display(money_cols(cdmx_metricas[["anio", "metrica", "n", "n_ponderado", "media", "p25", "mediana", "p75", "p90", "p95", "gini_0_100"]]))

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=cdmx_metricas, x="anio", y="mediana", hue="metrica", marker="o", ax=ax)
ax.set_title("CDMX: medianas ponderadas por métrica")
ax.set_xlabel("Año")
ax.set_ylabel("Pesos nominales trimestrales")
ax.set_xticks(YEARS)
ax.legend(title="Métrica", fontsize=8)
display(fig)
plt.close(fig)

## Brechas territoriales

Aquí se separa la idea de desigualdad interna y brecha territorial:

- desigualdad interna: Gini dentro de cada región, tamaño de localidad o estrato;
- brecha territorial: diferencia o razón entre medianas de grupos territoriales dentro del mismo año.

Esta no es una descomposición formal del Gini; es una lectura descriptiva para orientar la tesina.

In [ ]:
def median_gap(summary_df, dim):
    rows = []
    for year, g in summary_df.groupby("anio"):
        hi = g.loc[g["mediana"].idxmax()]
        lo = g.loc[g["mediana"].idxmin()]
        rows.append({
            "anio": int(year),
            "dimension": dim,
            "grupo_mayor_mediana": hi[dim],
            "mediana_mayor": hi["mediana"],
            "grupo_menor_mediana": lo[dim],
            "mediana_menor": lo["mediana"],
            "razon_mediana": hi["mediana"] / lo["mediana"],
            "brecha_mediana": hi["mediana"] - lo["mediana"],
        })
    return pd.DataFrame(rows)

brechas = pd.concat([
    median_gap(gini_region, "region_banxico"),
    median_gap(gini_tam_loc, "tam_loc_desc"),
    median_gap(gini_est_socio, "est_socio_desc"),
], ignore_index=True)
display(money_cols(brechas.sort_values(["dimension", "anio"])))

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=brechas, x="anio", y="razon_mediana", hue="dimension", marker="o", ax=ax)
ax.set_title("Brecha territorial: razón entre mayor y menor mediana")
ax.set_xlabel("Año")
ax.set_ylabel("Razón de medianas")
ax.set_xticks(YEARS)
ax.legend(title="Dimensión")
display(fig)
plt.close(fig)

## Grandes urbes

No se encontró en el proyecto una delimitación metropolitana oficial ya incorporada. Por lo tanto, este notebook **no aproxima grandes urbes con municipios sueltos**.

Acción tomada: dejar como enriquecimiento futuro la incorporación de una delimitación oficial. Para esta etapa solo se exploran variables existentes:

- `tam_loc_desc` para tamaño de localidad;
- `est_socio_desc` para estrato socioeconómico.

Estas variables no se etiquetan como zonas metropolitanas ni como marginalidad.

In [ ]:
candidate_files = list((ROOT / "reports").glob("*.md")) + list((ROOT / "docs").glob("*.md")) + [ROOT / "README.md"]
terms = ["metropol", "zona metropolitana", "delimitacion metropolitana", "delimitación metropolitana", "zmvm"]
rows = []
for path in candidate_files:
    if not path.exists():
        continue
    text = path.read_text(encoding="utf-8", errors="ignore").lower()
    for term in terms:
        if term in text:
            rows.append({"archivo": str(path.relative_to(ROOT)), "termino": term})
metropolitana = pd.DataFrame(rows)
if metropolitana.empty:
    display(Markdown("No se encontraron delimitaciones metropolitanas oficiales en los documentos del proyecto."))
else:
    display(metropolitana)

## Resumen de hallazgos

- El Gini nacional propio con ingreso corriente total del hogar baja de 43.83 en 2018 a 40.06 en 2024.
- La comparación con Banxico conserva el patrón general de reducción, pero queda por debajo del benchmark 2018-2022; la mayor discrepancia es Sur 2020, con -3.22 puntos.
- En 2024, Norte tiene la mayor mediana regional de ingreso corriente del hogar y Sur la menor.
- La brecha de mediana 2024 es de 1.77 veces entre Norte y Sur; de 2.00 veces entre localidades de 100,000+ habitantes y localidades menores de 2,500; y de 3.28 veces entre estrato Alto y Bajo.
- CDMX muestra aumento nominal fuerte de medianas entre 2020 y 2024, pero la lectura temporal debe esperar deflactación si se quiere hablar de poder adquisitivo.
- No hay delimitación metropolitana oficial incorporada; grandes urbes queda pendiente de enriquecimiento externo.